# 01 - Coleta dos Dados de Transporte (GTFS + OSM)

Baixa e explora os dados de transporte público do Grande Recife (GTFS) e do OpenStreetMap.

**GTFS** (General Transit Feed Specification) é um formato padrão com arquivos `.txt` dentro de um ZIP:
- `stops.txt` → paradas de ônibus (latitude, longitude, nome)
- `routes.txt` → linhas de ônibus
- `trips.txt` → viagens de cada linha
- `stop_times.txt` → horários de cada parada

**Fonte:** https://www.granderecife.pe.gov.br/gtfs/

In [ ]:
import zipfile
import os
import urllib.request
import pandas as pd
import geopandas as gpd
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt

In [ ]:
# Localiza a raiz do projeto (onde está o .git)
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()

ROOT     = find_root()
RAW_DIR  = ROOT / 'data' / 'raw' / 'gtfs'
ZIP_PATH = RAW_DIR / 'gtfs_grande_recife.zip'

# URL oficial do GTFS do Grande Recife Consórcio
# Se der erro de download, acesse o site e baixe manualmente para data/raw/gtfs/
GTFS_URL = 'https://www.granderecife.pe.gov.br/gtfs/gtfs.zip'

os.makedirs(RAW_DIR, exist_ok=True)
print(f'ROOT: {ROOT}')
print(f'Diretório de destino: {RAW_DIR}')

In [ ]:
# Baixa o ZIP do GTFS (se ainda não existir)
if not ZIP_PATH.exists():
    print(f'Baixando GTFS de: {GTFS_URL}')
    print('Isso pode levar alguns segundos...')
    try:
        urllib.request.urlretrieve(GTFS_URL, ZIP_PATH)
        print(f'ZIP salvo em: {ZIP_PATH}')
    except Exception as e:
        print(f'ERRO no download automático: {e}')
        print()
        print('>> Acesse https://www.granderecife.pe.gov.br/gtfs/ manualmente')
        print(f'>> Salve o arquivo como: {ZIP_PATH}')
        raise
else:
    print(f'GTFS já baixado: {ZIP_PATH}')

# Lista os arquivos dentro do ZIP
print('\nArquivos dentro do ZIP:')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for nome in zf.namelist():
        print(' -', nome)

In [ ]:
# Extrai o ZIP
EXTRACT_DIR = RAW_DIR / 'gtfs_extraido'
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print(f'Extraído em: {EXTRACT_DIR}')
print('Arquivos:', os.listdir(EXTRACT_DIR))

In [ ]:
# Lê as paradas de ônibus (stops.txt)
# Colunas importantes: stop_id, stop_name, stop_lat, stop_lon
stops = pd.read_csv(EXTRACT_DIR / 'stops.txt')
print(f'Total de paradas no GTFS: {len(stops)}')
print('Colunas:', stops.columns.tolist())
display(stops.head(5))

In [ ]:
# Lê as linhas (routes.txt)
routes = pd.read_csv(EXTRACT_DIR / 'routes.txt')
print(f'Total de linhas: {len(routes)}')
display(routes.head(5))

In [ ]:
# Converte as paradas para GeoDataFrame (tabela com geometria de pontos)
gdf_stops = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
    crs='EPSG:4326'   # coordenadas WGS84 (latitude/longitude padrão)
)

print(f'GeoDataFrame de paradas criado: {len(gdf_stops)} linhas')
print('CRS:', gdf_stops.crs)
display(gdf_stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'geometry']].head(5))

In [ ]:
# Filtra apenas paradas dentro da área de Recife
# Caixa delimitadora (bounding box) aproximada de Recife:
# Longitude: -35.05 a -34.87 | Latitude: -8.18 a -7.93
mask = (
    (gdf_stops['stop_lat'] >= -8.18) & (gdf_stops['stop_lat'] <= -7.93) &
    (gdf_stops['stop_lon'] >= -35.05) & (gdf_stops['stop_lon'] <= -34.87)
)
gdf_stops_recife = gdf_stops[mask].copy()
print(f'Paradas dentro da área de Recife: {len(gdf_stops_recife)}')

In [ ]:
# Plota as paradas no mapa
fig, ax = plt.subplots(figsize=(10, 10))
gdf_stops_recife.plot(ax=ax, markersize=2, color='steelblue', alpha=0.6)
ax.set_title('Paradas de Ônibus em Recife (GTFS)', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()
print(f'Total de paradas plotadas: {len(gdf_stops_recife)}')

In [ ]:
# Salva as paradas filtradas para uso no próximo notebook
OUT_STOPS = ROOT / 'data' / 'processed' / 'recife_paradas.geojson'
os.makedirs(OUT_STOPS.parent, exist_ok=True)
gdf_stops_recife.to_file(OUT_STOPS, driver='GeoJSON')
print(f'Paradas salvas em: {OUT_STOPS}')

In [ ]:
# Baixa a malha viária de Recife via OSMnx (OpenStreetMap)
# Necessário para análise real de acessibilidade a pé até os pontos de ônibus
import osmnx as ox

OUT_MALHA = ROOT / 'data' / 'processed' / 'recife_malha_viaria.geojson'

if not OUT_MALHA.exists():
    print('Baixando malha viária de Recife via OSMnx (pode demorar ~1-2 min)...')
    G = ox.graph_from_place('Recife, Pernambuco, Brasil', network_type='walk')
    _, edges = ox.graph_to_gdfs(G)
    edges_simples = edges[['geometry']].reset_index(drop=True)
    edges_simples.to_file(OUT_MALHA, driver='GeoJSON')
    print(f'Malha viária salva em: {OUT_MALHA}')
    print(f'Total de segmentos de rua: {len(edges_simples)}')
else:
    print(f'Malha viária já existe: {OUT_MALHA}')
    edges_simples = gpd.read_file(OUT_MALHA)
    print(f'Total de segmentos de rua: {len(edges_simples)}')